# **1. Dataset Description**

The dataset provided contains a comprehensive collection of residential real estate listings in Bengaluru, which we will use to build a regression model to predict property market prices. Understanding the structure and composition of this data is the foundational step in our analysis and feature engineering process.

| Column | Description |
| :--- | :--- |
| **price** | **(Target)** The market value of the property in lakhs (INR). |
| **location** | The area or locality of the property within Bengaluru. |
| **size** | The number of bedrooms, halls, and kitchens (BHK) or bedroom count. |
| **total_sqft** | The total surface area of the property in square feet. |
| **bath** | The total number of bathrooms in the property. |
| **balcony** | The total number of balconies available. |

### Dataset Overview
*   **Total Listings:** Records covering diverse residential configurations across various Bengaluru localities.
*   **Purpose:** Development of a price prediction model to assist in property valuation and real estate market analysis.
*   **Attributes:** The dataset encompasses key spatial and structural specifications including property size, square footage, and amenity counts, allowing for a multifaceted approach to valuation modeling.

## Live Demo
[![Streamlit App](https://static.streamlit.io/badges/streamlit_badge_black_white.svg)](https://donna7-house-price-predictor.streamlit.app)

# **2. Import Libraries**

In [145]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pickle

# **3. Import dataset**

In [146]:
df = pd.read_csv('Bengaluru_House_Data.csv')

In [147]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [148]:
df.shape

(13320, 9)

In [149]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


### Data Quality & Observations
The dataset contains a total of 13,320 entries. Upon initial inspection, we identified significant null values, which require careful handling during the preprocessing stage:

*   **Society Column:** This feature has the highest volume of missing data (only 7,818 non-null entries), indicating that many listings do not belong to a registered society.
*   **Other Missing Data:** Minor gaps were observed in the `size`, `bath`, `balcony`, and `location` columns.

**Preprocessing Strategy:**
We will address these gaps by applying domain-specific imputation techniques—such as filling missing values with the median or mode—or by dropping columns where the missing information is too high to provide predictive value for the model.

# **4. Data cleaning**

In [150]:
for column in df.columns:
  print(df[column].value_counts())
  print('='*50)

area_type
Super built-up  Area    8790
Built-up  Area          2418
Plot  Area              2025
Carpet  Area              87
Name: count, dtype: int64
availability
Ready To Move    10581
18-Dec             307
18-May             295
18-Apr             271
18-Aug             200
                 ...  
16-Oct               1
17-Jan               1
16-Nov               1
16-Jan               1
14-Jul               1
Name: count, Length: 81, dtype: int64
location
Whitefield                         540
Sarjapur  Road                     399
Electronic City                    302
Kanakpura Road                     273
Thanisandra                        234
                                  ... 
3rd Stage Raja Rajeshwari Nagar      1
Chuchangatta Colony                  1
Electronic City Phase 1,             1
Chikbasavanapura                     1
Abshot Layout                        1
Name: count, Length: 1305, dtype: int64
size
2 BHK         5199
3 BHK         4310
4 Bedroom      826
4 BH

In [151]:
df.isna().sum()

,0
area_type,0
availability,0
location,1
size,16
society,5502
total_sqft,0
bath,73
balcony,609
price,0


In [152]:
df.drop(columns=['availability','society','balcony','area_type'], inplace=True)

*   `society`: This column contains a high proportion of null values (approx. 41% missing). In a real-world scenario, missing data of this magnitude often introduces more bias than predictive signal, making it unreliable for training a stable model.
*   `Balcony`: After testing the model's feature importance, `balcony` proved to be statistically insignificant. We observed that balcony counts are often highly correlated with the property size or total square footage, providing redundant information rather than unique insight into the price.
*   `Area Type` & `Availability`: These features showed minimal correlation with the target variable (`price`). Properties are primarily valued based on location, size, and utility rather than the specific type of area designation (e.g., Super built-up vs. Plot) or the availability date, which often acts more as a descriptive metadata tag rather than a price driver.

By removing these non-impactful columns, we have streamlined the dataset, reduced the dimensionality of our feature space, and focused the model on the most significant predictors of property valuation in Bengaluru.

In [153]:
df.describe()

,bath,price
count,13247.000000,13320.000000
mean,2.692610,112.565627
std,1.341458,148.971674
min,1.000000,8.000000
25%,2.000000,50.000000
50%,2.000000,72.000000
75%,3.000000,120.000000
max,40.000000,3600.000000


In [154]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13319 non-null  object 
 1   size        13304 non-null  object 
 2   total_sqft  13320 non-null  object 
 3   bath        13247 non-null  float64
 4   price       13320 non-null  float64
dtypes: float64(2), object(3)
memory usage: 520.4+ KB


In [155]:
df['location'].value_counts()

,count
location,
Whitefield,540
Sarjapur Road,399
Electronic City,302
Kanakpura Road,273
Thanisandra,234
...,...
3rd Stage Raja Rajeshwari Nagar,1
Chuchangatta Colony,1
"Electronic City Phase 1,",1


In [156]:
df['location'] = df['location'].fillna('Sarjapur Road')

since there is only one null value in `location`, we will impute the value with the second most frequent value.

In [157]:
df['size'].value_counts()

,count
size,
2 BHK,5199
3 BHK,4310
4 Bedroom,826
4 BHK,591
3 Bedroom,547
1 BHK,538
2 Bedroom,329
5 Bedroom,297
6 Bedroom,191


In [158]:
df['size'] = df['size'].fillna('2 BHK')

filling the missing values with mode (2 BHK)

In [159]:
df['bath'].value_counts()

,count
bath,
2.0,6908
3.0,3286
4.0,1226
1.0,788
5.0,524
6.0,273
7.0,102
8.0,64
9.0,43


In [160]:
df['bath'] = df['bath'].fillna(df['bath'].median())

filling the missing values in `bath` with the median

In [161]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13320 non-null  object 
 1   size        13320 non-null  object 
 2   total_sqft  13320 non-null  object 
 3   bath        13320 non-null  float64
 4   price       13320 non-null  float64
dtypes: float64(2), object(3)
memory usage: 520.4+ KB


all mising values are covered

In [162]:
df['bhk'] = df['size'].str.split().str.get(0).astype(int)

a new column `bhk` is made to store the numbers in `size` column after splitting with spaces

In [163]:
df[df.bhk > 20]

,location,size,total_sqft,bath,price,bhk
1718,2Electronic City Phase II,27 BHK,8000,27.0,230.0,27
4684,Munnekollal,43 Bedroom,2400,40.0,660.0,43


a house containing more than 20 BHK or bedrooms is probably an outlier

In [164]:
df['total_sqft'].value_counts()

,count
total_sqft,
1200,843
1100,221
1500,205
2400,196
600,180
...,...
2920,1
5665.84,1
1369,1


`total_sqft` contains ranges so we will take the mean for this ranges

In [165]:
df[df['total_sqft'].isna()]

,location,size,total_sqft,bath,price,bhk


In [166]:
def convertRange(x):
    # 1. Handle ranges (already working)
    temp = x.split('-')
    if len(temp) == 2:
        return (float(temp[0]) + float(temp[1])) / 2

    # 2. Try simple conversion
    try:
        return float(x)
    except:
        # 3. If that fails, try to extract numbers from units (e.g., "1200Sq. Ft")
        # This regex finds the first sequence of numbers (including decimals)
        match = re.search(r'(\d+\.?\d*)', x)
        if match:
            return float(match.group(1))
        return None

In [167]:
df['total_sqft'] = df['total_sqft'].apply(convertRange)

In [168]:
df[df['total_sqft'].isna()]

,location,size,total_sqft,bath,price,bhk


In [169]:
df.head()

,location,size,total_sqft,bath,price,bhk
0,Electronic City Phase II,2 BHK,1056.0,2.0,39.07,2
1,Chikka Tirupathi,4 Bedroom,2600.0,5.0,120.00,4
2,Uttarahalli,3 BHK,1440.0,2.0,62.00,3
3,Lingadheeranahalli,3 BHK,1521.0,3.0,95.00,3
4,Kothanur,2 BHK,1200.0,2.0,51.00,2


### Price Per Sqaure feet

In [170]:
df['price_per_sqft'] = df['price']*100000/df['total_sqft']

* Property prices in the dataset are in lakhs, and sizes vary significantly. Calculating the price per square foot allows us to compare the relative value of properties regardless of their total size, acting as a crucial indicator for identifying market outliers.
* This metric is **used exclusively for outlier detection and cleaning**. By analyzing the distribution of price per square foot within each location, we can identify and remove properties that are priced unrealistically high or low compared to the local market mean. Once the data is cleaned and the outliers are removed, we rely on the primary features for model training.

In [171]:
df['price_per_sqft'].isna().sum()

np.int64(0)

In [172]:
df.describe()

,total_sqft,bath,price,bhk,price_per_sqft
count,13320.000000,13320.000000,13320.000000,13320.000000,1.332000e+04
mean,1555.971707,2.688814,112.565627,2.802778,1.234333e+04
std,1238.902448,1.338754,148.971674,1.294496,2.256394e+05
min,1.000000,1.000000,8.000000,1.000000,2.678298e+02
25%,1100.000000,2.000000,50.000000,2.000000,4.271723e+03
50%,1275.000000,2.000000,72.000000,3.000000,5.444545e+03
75%,1679.250000,3.000000,120.000000,3.000000,7.335907e+03
max,52272.000000,40.000000,3600.000000,43.000000,1.600000e+07


*   **Extreme Price and Sqft Outliers:** The `max` values for `total_sqft` (52,272) and `price_per_sqft` (16,000,000) remain exponentially higher than the 75th percentile (1,679 and 7,335, respectively). This confirms that extreme statistical anomalies are still present and would significantly skew a linear regression model.
*   **Implausible Property Configurations:** The `max` values for `bath` (40) and `bhk` (43) persist as extreme outliers, indicating that despite data consistency improvements, we still need to filter for residential-standard configurations to ensure model accuracy.
*   **Distribution Skew:** For both `price` and `total_sqft`, the `mean` continues to track notably higher than the `50%` (median) value. This confirms a right-skewed distribution where a small minority of exceptionally large or high-priced properties continue to inflate the average.

In [173]:
df['location'] = df['location'].apply(lambda x: x.strip())
location_count = df['location'].value_counts()

We applied `.strip()` to the `location` column to remove accidental leading or trailing whitespace, ensuring that entries like "Electronic City" and " Electronic City " are correctly identified as the same location. Following this, we used `.value_counts()` to aggregate the frequency of each location.

**Purpose:** This process is essential for grouping properties by area and identifying "low-frequency" locations. We will use this information to group these rare locations into an **"Other"** category, which reduces dimensionality and prevents the model from overfitting on areas with insufficient data.

In [174]:
location_count_less_10 = location_count[location_count <= 10]
location_count_less_10

,count
location,
1st Block Koramangala,10
Dairy Circle,10
Nagadevanahalli,10
Sadashiva Nagar,10
Naganathapura,10
...,...
Xavier Layout,1
Ramanagara Channapatna,1
Maheswari Nagar,1


In [175]:
df['location'] = df['location'].apply(lambda x: 'other' if x in location_count_less_10 else x)

In [176]:
df['location'].value_counts()

,count
location,
other,2886
Whitefield,541
Sarjapur Road,399
Electronic City,304
Kanakpura Road,273
...,...
Tindlu,11
Marsur,11
2nd Phase Judicial Layout,11


### Outlier Detection and Removal

In [177]:
df.describe()

,total_sqft,bath,price,bhk,price_per_sqft
count,13320.000000,13320.000000,13320.000000,13320.000000,1.332000e+04
mean,1555.971707,2.688814,112.565627,2.802778,1.234333e+04
std,1238.902448,1.338754,148.971674,1.294496,2.256394e+05
min,1.000000,1.000000,8.000000,1.000000,2.678298e+02
25%,1100.000000,2.000000,50.000000,2.000000,4.271723e+03
50%,1275.000000,2.000000,72.000000,3.000000,5.444545e+03
75%,1679.250000,3.000000,120.000000,3.000000,7.335907e+03
max,52272.000000,40.000000,3600.000000,43.000000,1.600000e+07


In [178]:
df = df[((df['total_sqft'])/df['bhk']) >= 300]
df.describe()

,total_sqft,bath,price,bhk,price_per_sqft
count,12538.000000,12538.000000,12538.000000,12538.000000,12538.000000
mean,1594.853464,2.560217,111.390611,2.651460,6303.772400
std,1261.221561,1.079405,152.041719,0.978385,4161.151656
min,300.000000,1.000000,8.440000,1.000000,267.829813
25%,1116.000000,2.000000,49.000000,2.000000,4210.526316
50%,1300.000000,2.000000,70.000000,3.000000,5294.117647
75%,1700.000000,3.000000,115.000000,3.000000,6916.666667
max,52272.000000,16.000000,3600.000000,16.000000,176470.588235


### Outlier Removal: Domain-Specific Filtering
We applied a threshold filter to keep only those properties where the square footage per bedroom (`total_sqft / bhk`) is at least **300 square feet**.

**Why we do this:**
In the Bengaluru real estate market, a standard residential bedroom is typically expected to be at least 300 square feet. This step is a **domain-specific cleaning technique** used to eliminate "impossible" data points.

In [179]:
df.shape

(12538, 7)

In [180]:
df['price_per_sqft'].describe()

,price_per_sqft
count,12538.000000
mean,6303.772400
std,4161.151656
min,267.829813
25%,4210.526316
50%,5294.117647
75%,6916.666667
max,176470.588235


The descriptive statistics for `price_per_sqft` indicate a severe right-skewed distribution. While the 75th percentile is approximately **6,917**, the maximum value reaches an extreme of **176,470**, so the outliers here must be handled

In [181]:
def remove_outliers_sqft(df):
    def filter_group(x):
        m = x.price_per_sqft.mean()
        st = x.price_per_sqft.std()
        # Returns only the rows that fall within the range
        return x[(x.price_per_sqft > (m - st)) & (x.price_per_sqft <= (m + st))]

    return df.groupby('location', group_keys=False).apply(filter_group)

This function cleans the dataset by grouping properties by `location` and discarding extreme price outliers that fall outside one standard deviation of the local average, ensuring the model learns only from realistic market trends.

In [182]:
df = remove_outliers_sqft(df)

/tmp/ipykernel_835/1024085483.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('location', group_keys=False).apply(filter_group)


In [183]:
df.describe()

,total_sqft,bath,price,bhk,price_per_sqft
count,10358.000000,10358.000000,10358.000000,10358.000000,10358.000000
mean,1509.062146,2.473450,91.372831,2.576366,5660.998274
std,880.163072,0.982624,86.423870,0.900854,2270.770637
min,300.000000,1.000000,10.000000,1.000000,1250.000000
25%,1109.250000,2.000000,49.000000,2.000000,4240.695885
50%,1286.000000,2.000000,67.000000,2.000000,5175.519668
75%,1650.000000,3.000000,100.000000,3.000000,6431.808627
max,30400.000000,16.000000,2200.000000,16.000000,24509.803922


In [184]:
def bhk_outlier_remover(df):
    exclude_indices = []

    for location, location_df in df.groupby('location'):
        bhk_stats = {}
        # Pre-calculate stats per BHK for this location
        for bhk, bhk_df in location_df.groupby('bhk'):
            bhk_stats[bhk] = {
                'mean': bhk_df.price_per_sqft.mean(),
                'count': bhk_df.shape[0]
            }

        # Identify indices to exclude based on the previous BHK's mean
        for bhk, bhk_df in location_df.groupby('bhk'):
            prev_bhk = bhk_stats.get(bhk - 1)
            if prev_bhk and prev_bhk['count'] > 5:
                # Add indices where current BHK is cheaper than the mean of the smaller BHK
                exclude_indices.extend(bhk_df[bhk_df.price_per_sqft < prev_bhk['mean']].index.values)

    return df.drop(exclude_indices, axis='index')

This function removes data errors by dropping properties where a larger unit costs less per square foot than the average price of a smaller unit in the same location, ensuring the dataset maintains logical pricing consistency.

In [185]:
df = bhk_outlier_remover(df)

In [186]:
df.shape

(7416, 7)

In [187]:
df

,location,size,total_sqft,bath,price,bhk,price_per_sqft
210,1st Block Jayanagar,4 BHK,2850.0,4.0,428.0,4,15017.543860
848,1st Block Jayanagar,3 BHK,1630.0,3.0,194.0,3,11901.840491
2152,1st Block Jayanagar,3 BHK,1875.0,2.0,235.0,3,12533.333333
7138,1st Block Jayanagar,3 BHK,1200.0,2.0,130.0,3,10833.333333
7358,1st Block Jayanagar,2 BHK,1235.0,2.0,148.0,2,11983.805668
...,...,...,...,...,...,...,...
13238,other,2 BHK,1200.0,2.0,70.0,2,5833.333333
13245,other,1 BHK,1800.0,1.0,200.0,1,11111.111111
13285,other,2 BHK,1353.0,2.0,110.0,2,8130.081301
13291,other,1 Bedroom,812.0,1.0,26.0,1,3201.970443


In [188]:
df.drop(columns=['size','price_per_sqft'], inplace=True)

We drop these columns because `size` is no longer needed after extracting the `bhk` data, and `price_per_sqft` is removed to prevent data leakage, as it would reveal the target price variable to the model during training and it was used to handle outliers only

### Cleaned Data

In [189]:
df.head()

,location,total_sqft,bath,price,bhk
210,1st Block Jayanagar,2850.0,4.0,428.0,4
848,1st Block Jayanagar,1630.0,3.0,194.0,3
2152,1st Block Jayanagar,1875.0,2.0,235.0,3
7138,1st Block Jayanagar,1200.0,2.0,130.0,3
7358,1st Block Jayanagar,1235.0,2.0,148.0,2


In [190]:
df.to_csv('Cleaned_data.csv')

# **5. Data Preprocessing**

## **5.1. Declare feature vector and target variable**

In [191]:
X = df.drop('price', axis=1)
y = df['price']

## 5.2. Data Splitting: Training and Testing Sets

In [192]:
from sklearn.model_selection import train_test_split

In [193]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [194]:
print(X_train.shape)
print(X_test.shape)

(5932, 4)
(1484, 4)


## 5.3. Encoding Categorical Data with OneHotEncoder

In [195]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import make_column_transformer

In [196]:
column_trans = make_column_transformer(
    (OneHotEncoder(sparse_output=False), ['location']),
    remainder='passthrough'
)

In [197]:
scaler = StandardScaler()

# **6. Model Building**

### 6.1. Applying Linear Regression

In [198]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import make_pipeline

In [199]:
lr = LinearRegression()

In [200]:
pipe = make_pipeline(column_trans, scaler, lr)

In [201]:
pipe.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(sparse_output=False),
                                                  ['location'])])),
                ('standardscaler', StandardScaler()),
                ('linearregression', LinearRegression())])

In [202]:
y_pred_lr = pipe.predict(X_test)

In [203]:
r2_score(y_test, y_pred_lr) # No Regularization

0.801215127336242

### 6.2. Applying Lasso

In [204]:
lasso = Lasso()

In [205]:
pipe = make_pipeline(column_trans, scaler, lasso)

In [206]:
pipe.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(sparse_output=False),
                                                  ['location'])])),
                ('standardscaler', StandardScaler()), ('lasso', Lasso())])

In [207]:
y_pred_lasso = pipe.predict(X_test)

In [208]:
r2_score(y_test, y_pred_lr)

0.801215127336242

### 6.3. Applying Ridge

In [209]:
ridge = Ridge()

In [210]:
pipe = make_pipeline(column_trans, scaler, ridge)

In [211]:
pipe.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(sparse_output=False),
                                                  ['location'])])),
                ('standardscaler', StandardScaler()), ('ridge', Ridge())])

In [212]:
y_pred_ridge = pipe.predict(X_test)

In [213]:
r2_score(y_test, y_pred_ridge)

0.8011823609782746

# **7. Deployment Preperation**

In [214]:
data = pd.read_csv('Cleaned_data.csv')

In [215]:
data.head()

,Unnamed: 0,location,total_sqft,bath,price,bhk
0,210,1st Block Jayanagar,2850.0,4.0,428.0,4
1,848,1st Block Jayanagar,1630.0,3.0,194.0,3
2,2152,1st Block Jayanagar,1875.0,2.0,235.0,3
3,7138,1st Block Jayanagar,1200.0,2.0,130.0,3
4,7358,1st Block Jayanagar,1235.0,2.0,148.0,2


In [216]:
import pickle

# Save the model pipeline
pickle.dump(pipe, open('RidgeModel.pkl', 'wb'))

# Save the dataframe
pickle.dump(df, open('data.pkl', 'wb'))